# 05-2 · The objection parser

| | |
|---|---|
| **What it does** | turns what a diner actually types ("ugh, not Thai again") into a label the mediator's maths can consume |
| **Proves** | the free-text -> reaction step is a measured component with a known error rate, not a hand-wave |
| **Writes** | `figdata/table6_parser.csv` |
| **Needs** | `GROQ_API_KEY` to re-run the parse; without one it loads the committed predictions |

### Where this sits

`05-mediate/01` showed the mediator reweighting on reaction labels that were
handed to it. Those labels have to come from somewhere. In a real chat they
come from a sentence a person typed — and that conversion is the weakest,
most attackable link in the whole loop, so it gets measured like any other
component.

```
   "ugh, not Thai again"          <- what a person types
            |
            v   parse_objection()          <- THIS NOTEBOOK
   {member: Kyle, label: strong_dislike, confidence: 0.9}
            |
            v   REACTION_GRADIENT
          g = 0.5                          <- a number the maths can use
            |
            v   Mediator.update_weights()
   w_Kyle goes up, the next pick shifts toward him
```

In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
os.chdir(_root)   # data/ and figdata/ paths below are repo-relative

In [2]:
import nbio
nbio.bootstrap()

repo     : /project/painting/local-projects-to-run/AnacodicAI-Labs/code/ai-in-food/research_fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/AnacodicAI-Labs/code/ai-in-food/research_fair-mediator')

## Step 1 — the four labels, and what each one costs

A label is only meaningful because it maps to a number. `REACTION_GRADIENT`
is the single place that conversion happens, so the "how much does a
complaint move the recommendation" decision lives in exactly one line of the
codebase rather than being scattered through the loop.

In [3]:
print("valid labels:", nbio.VALID_LABELS)
print()
for label, g in nbio.REACTION_GRADIENT.items():
    bar = "#" * int(g * 20)
    print(f"  {label:16s} g = {g:<4} {bar}")
print()
print("A veto (1.5) moves the weights 3x as hard as a strong_dislike (0.5),")
print("and 'accept' (0.0) leaves them untouched -- agreeing costs you nothing.")

valid labels: ('accept', 'reject', 'strong_dislike', 'veto')

  accept           g = 0.0  
  reject           g = 0.2  ####
  strong_dislike   g = 0.5  ##########
  veto             g = 1.5  ##############################

A veto (1.5) moves the weights 3x as hard as a strong_dislike (0.5),
and 'accept' (0.0) leaves them untouched -- agreeing costs you nothing.


## Step 2 — the prompt, frozen

Same rule as the judge in `07-judge/01`: the prompt is committed under a
version string **before** any accuracy number was measured. Tuning a prompt
until the accuracy looks good, then reporting that accuracy, is measuring the
tuning rather than the parser.

In [4]:
print("prompt version:", nbio.OBJECTION_PROMPT_VERSION)
print()
print(nbio.OBJECTION_PROMPT)

prompt version: v1

You classify one person's reaction to a restaurant suggestion.

Reply with ONLY a JSON object, no prose, no markdown fence:
{"member": "<name>", "label": "<label>", "confidence": <0.0-1.0>}

label must be exactly one of: accept, reject, strong_dislike, veto
  accept          they are happy with the suggestion, including reluctant or
                  conditional agreement ("fine if they have vegan options")
  reject          mild disagreement, a preference for something else
  strong_dislike  strong negative feeling. SARCASM counts as strong_dislike
                  ("oh great, sushi again" is NOT acceptance)
  veto            they explicitly veto/block it, OR state a hard safety
                  constraint that rules the option out (allergy, diet)

member must be the person whose PREFERENCE is being expressed, which is not
always the speaker. "Cara won't eat that" expresses Cara's preference, so
member is Cara. Use "SELF" when the speaker is talking about themsel

## Step 3 — what happens when the parse fails

This is the part that matters most, and it is a **safety** decision rather
than an accuracy one.

If the parser cannot understand a sentence, it must not guess `accept`.
Silence is not consent: a malformed complaint that quietly becomes agreement
means the group proceeds past an objection nobody registered. So an
unparseable reply degrades to **`reject` at confidence 0.0** — the mediator
still hears "something was wrong here", and the confidence being 0 marks it
as a parse failure rather than a mild complaint.

An invented member name is different: that is not a degraded answer, it is a
wrong one, and it raises instead.

In [5]:
MEMBERS = ["Harper", "Abigail", "Kyle"]

def fake_client(reply):                       # a parser that returns whatever we hand it
    return type("X", (), {"chat": type("c", (), {"completions": type("x", (), {
        "create": staticmethod(lambda **k: type("R", (), {"choices": [type("C", (), {
            "message": type("M", (), {"content": reply})()})()]})())})})})

for name, reply in [
        ("garbage out",        "I think Kyle is maybe unhappy?"),
        ("invalid label",      '{"member":"Kyle","label":"furious","confidence":0.9}'),
        ("empty reply",        ""),
]:
    p = nbio.parse_objection("whatever", MEMBERS, client=fake_client(reply), model="m")
    print(f"  {name:16s} -> label={p['label']:<8} confidence={p['confidence']}")
    assert p["label"] != "accept", "a failed parse must never become consent"

print()
try:
    nbio.parse_objection("x", MEMBERS, model="m",
                         client=fake_client('{"member":"Zoltan","label":"veto","confidence":1.0}'))
except ValueError as e:
    print("  invented member  -> ValueError:", e)
print()
print("-> no failure path can produce silent agreement.")

  garbage out      -> label=reject   confidence=0.0
  invalid label    -> label=reject   confidence=0.0
  empty reply      -> label=reject   confidence=0.0

  invented member  -> ValueError: parse_objection: model named 'Zoltan', who is not in the group ['Harper', 'Abigail', 'Kyle']. Refusing to reweight an unknown diner.

-> no failure path can produce silent agreement.


## Step 4 — the labelled set

100 reactions, hand-labelled, deliberately stratified so the hard cases cannot
hide inside an average. `plain` is the easy majority; the other four
categories are the ones that break parsers.

In [6]:
import pandas as pd
gold = pd.read_csv("data/objections/labelled_reactions.csv")
print(gold.groupby("category").size().to_string())
print()
for cat in ["plain", "other_member", "hedge", "conditional", "sarcasm"]:
    ex = gold[gold.category == cat].iloc[0]
    print(f"  {cat:14s} \"{ex.text}\"")
    print(f"  {'':14s}  -> {ex.gold_label} (about: {ex.gold_member})")

category
conditional      8
hedge           10
other_member    12
plain           62
sarcasm          8

  plain          "sounds good to me"
                  -> accept (about: SELF)
  other_member   "Cara won't eat that"
                  -> reject (about: Cara)
  hedge          "I mean, it's fine I guess"
                  -> reject (about: SELF)
  conditional    "fine if they have something vegan"
                  -> accept (about: SELF)
  sarcasm        "oh great, sushi again"
                  -> strong_dislike (about: SELF)


> ### ⚠️ Single annotator
> Every gold label here was assigned by **one person**. With one annotator
> there is no way to tell a hard case from an idiosyncratic call, and the
> accuracy below is measured against one person's judgement rather than
> against agreement. A second annotator labelling blind, reported as Cohen's
> kappa, is an open task (tracker: MEDIATE-1) and this caveat belongs in the
> paper's limitations until it is done.

## Step 5 — run the parser

With a key this re-parses all 100 from scratch. Without one it loads the
committed predictions so the table still rebuilds.

In [7]:
from openai import OpenAI

KEY   = os.environ.get("GROQ_API_KEY", "")
LIVE  = bool(KEY)
MODEL = "openai/gpt-oss-20b"
client = OpenAI(api_key=KEY, base_url="https://api.groq.com/openai/v1") if LIVE else None
PRED = "data/objections/parser_predictions.csv"

if LIVE:
    meter = nbio.cost_meter(2.00)          # 100 calls; aborts long before this
    rows = []
    for _, r in gold.iterrows():
        members = [r.gold_member] if r.gold_member != "SELF" else []
        names = sorted(set(["SELF"] + members))
        p = nbio.parse_objection(r.text, names, client=client, model=MODEL)
        meter.record("parse", 0.0002)
        rows.append({**r.to_dict(), "pred_label": p["label"],
                     "pred_member": p["member"], "conf": p["confidence"]})
    pred = pd.DataFrame(rows); pred.to_csv(PRED, index=False)
    print(meter.report())
else:
    pred = pd.read_csv(PRED)
    print(f"loaded {len(pred)} committed predictions from {PRED}")

loaded 100 committed predictions from data/objections/parser_predictions.csv


## Step 6 — Table 6: accuracy, split by how hard the case is

Two accuracies, because two different things can go wrong:

- **label accuracy** — did it read the strength of the objection correctly?
- **member accuracy** — did it attribute the objection to the right person?

Getting the member wrong is the worse failure: it reweights toward someone who
never complained, which actively makes the recommendation less fair.

In [8]:
pred["label_ok"]  = pred.pred_label  == pred.gold_label
pred["member_ok"] = pred.pred_member == pred.gold_member

out = []
for cat, g in pred.groupby("category", sort=False):
    out.append({"category": cat, "n": len(g),
                "label_accuracy":  round(100 * g.label_ok.mean(), 1),
                "member_accuracy": round(100 * g.member_ok.mean(), 1)})
out.append({"category": "ALL", "n": len(pred),
            "label_accuracy":  round(100 * pred.label_ok.mean(), 1),
            "member_accuracy": round(100 * pred.member_ok.mean(), 1)})

t6 = pd.DataFrame(out)
t6.to_csv("figdata/table6_parser.csv", index=False)
t6

,category,n,label_accuracy,member_accuracy
0,plain,62,91.9,100.0
1,sarcasm,8,12.5,100.0
2,hedge,10,70.0,100.0
3,conditional,8,100.0,100.0
4,other_member,12,83.3,100.0
5,ALL,100,83.0,100.0


In [9]:
print("accuracy by category (label):\n")
for _, r in t6[t6.category != "ALL"].sort_values("label_accuracy").iterrows():
    print(f"  {r.category:14s} {r.label_accuracy:5.1f}%  {'#' * int(r.label_accuracy/4)}")
print(f"\n  {'ALL':14s} {t6[t6.category=='ALL'].label_accuracy.iloc[0]:5.1f}%")

accuracy by category (label):

  sarcasm         12.5%  ###
  hedge           70.0%  #################
  other_member    83.3%  ####################
  plain           91.9%  ######################
  conditional    100.0%  #########################

  ALL             83.0%


## Step 7 — the failure worth naming

Member attribution is **100% everywhere** — the parser always works out *who*
is objecting. Label accuracy is where it breaks, and it does not break evenly:

- `conditional` ("I'd go if they had a veg option") — **100%**, easy
- `plain` — **91.9%**
- `hedge` ("I mean, it's fine I guess") — **70%**, it reads hedges as agreement
- **`sarcasm` — 12.5%.** Effectively broken.

Sarcasm is not a rounding error, it is a systematic failure: "oh GREAT, Thai
again" reads as enthusiasm. The honest framing for the paper is that the
parser is reliable on literal text and unreliable on ironic text, and that the
`n = 8` sarcasm cell is far too small to quantify properly — it establishes
that the failure exists, not how big it is.

The degradation contract in Step 3 is what keeps this from being dangerous:
when the parser is unsure it produces `reject`, so the cost of failure is an
unnecessary extra round of mediation rather than an objection being dropped.

In [10]:
sar = pred[(pred.category == "sarcasm") & (~pred.label_ok)]
print("every sarcasm case the parser got wrong:\n")
for _, r in sar.iterrows():
    print(f"  \"{r.text}\"")
    print(f"     gold={r.gold_label:<15} predicted={r.pred_label:<15} conf={r.conf}")
print(f"\n{len(sar)} of {(pred.category=='sarcasm').sum()} sarcasm cases wrong.")
print("Note the confidences: the parser is not hedging, it is confidently wrong.")

every sarcasm case the parser got wrong:

  "wonderful, my favourite place in the world"
     gold=strong_dislike  predicted=accept          conf=1.0
  "brilliant, exactly what I was hoping for"
     gold=strong_dislike  predicted=accept          conf=0.95
  "oh fantastic, more noodles"
     gold=strong_dislike  predicted=accept          conf=0.95
  "yeah because that went so well last time"
     gold=strong_dislike  predicted=accept          conf=0.95
  "sure, if we want to queue for an hour"
     gold=reject          predicted=accept          conf=0.9
  "lovely, another salad bar"
     gold=reject          predicted=accept          conf=0.95
  "can't wait"
     gold=strong_dislike  predicted=accept          conf=0.95

7 of 8 sarcasm cases wrong.
Note the confidences: the parser is not hedging, it is confidently wrong.


## What this notebook establishes

1. The free-text -> label step has a **measured** error rate, split by case type.
2. It **cannot fail into consent** — every degradation path produces `reject`.
3. It attributes objections to the right person **100%** of the time.
4. It is **broken on sarcasm** (12.5%), which is stated rather than averaged away.
5. The labels are **single-annotator**, which caps what the accuracy means.

Next: `05-mediate/03` feeds these labels into the bounded mediation loop and
runs a whole conversation end to end.